# Tokens, contexto y coste: la factura de un LLM

**Facsímil 4 · La caja de herramientas** — capítulo 3 (tokens, coste, contexto y caché).

La primera sorpresa de cualquiera que pone un LLM en producción es la **factura**. No se paga por
«pregunta», se paga por **token**, y los tokens no se cuentan a ojo: un emoji puede valer varios, una
palabra rara se parte en trozos, una URL se desmenuza y el español gasta más que el inglés. En este
cuaderno mides, en tokens reales, lo que de verdad procesas y pagas; abres el tokenizador para ver
*cómo* trocea; calculas cuánto costaría clasificar diez mil tickets de soporte con varios modelos;
compruebas por qué una conversación larga se encarece sola; y entiendes por qué la **ventana de
contexto** y la **caché** deciden si tu sistema es viable o ruinoso. Saber estimar esto *antes* de
gastar es puro criterio de ingeniería.

### Qué vas a aprender
- Qué es un **token** y por qué tokens ≠ palabras ≠ caracteres (con un vistazo a cómo funciona BPE).
- A **abrir** una palabra y ver en qué subpalabras la parte el modelo.
- A **contar tokens** de cualquier texto y a estimar el **coste** de un volumen de trabajo.
- Por qué el **español** sale más caro que el inglés, y cómo se comportan otros idiomas y alfabetos.
- A separar el coste de **entrada** del de **salida**, y a comparar **varios modelos** por la misma tarea.
- Qué es la **ventana de contexto** y la **caché de prefijo**, por qué una **conversación larga** se
  encarece sola, y cuánto pesan estas cosas en la factura.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# En Colab:  !pip install tiktoken
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")   # el tokenizador de la familia GPT-4/3.5
def tk(texto): return enc.encode(texto)
def trozos(texto): return [enc.decode([t]) for t in tk(texto)]
print("Tokenizador cargado. Probemos con una frase:")
ej = "El café de la mañana me despierta."
print(f"  '{ej}'")
print(f"  -> {len(tk(ej))} tokens:", trozos(ej))


## 1. Qué es un token (y por qué no es una palabra)

Un modelo no procesa letras ni palabras: procesa **tokens**, trozos de texto de tamaño variable. El
método más común para trocear, **BPE** (*byte pair encoding*), parte de los caracteres y va fusionando
los pares más frecuentes hasta formar un vocabulario de subpalabras. El resultado: las palabras
comunes son **un** token («café», «de»), las raras se parten en varios, y los signos, espacios, cifras
y emojis cuentan también. Por eso *tokens ≠ palabras ≠ caracteres*. Veámoslo en una tabla con casos
variados: texto normal, una palabra inventada, código, emojis, una URL, un número largo y una fecha.


In [ ]:
ejemplos = [
    "Hola, ¿qué tal?",
    "supercalifragilisticoexpialidoso",
    "antidisestablishmentarianism",
    "def suma(a, b):\n    return a + b",
    "🙂🎉☕",
    "1234567890",
    "https://www.iaparagentecuriosa.686f6c61.dev/cap/3",
    "El 31 de mayo de 2026 a las 14:05h",
    "ESTO VA EN MAYÚSCULAS Y GRITANDO",
    "naïve café résumé jalapeño",
]
print(f"{'texto':<42} {'caract.':>8} {'palabras':>9} {'tokens':>7}")
for t in ejemplos:
    una_linea = t.replace(chr(10), "\\n")
    print(f"{una_linea[:40]!r:<42} {len(t):>8} {len(t.split()):>9} {len(tk(t)):>7}")


**Lo que se ve.** Una palabra corta y común suele ser 1 token; una rara se parte en muchos; el
código, los emojis y las URL se fragmentan (un emoji puede valer varios tokens porque se codifica en
varios bytes); los números largos se trocean en grupos de cifras; y hasta las **mayúsculas** o las
**tildes** pueden cambiar el recuento, porque «Hola» y «HOLA» no son el mismo token. Nada de esto se
adivina a ojo: por eso se cuenta. Y todo cuenta para la factura, que vemos enseguida.


## 2. Abrir una palabra: ver el troceado por dentro

La forma más rápida de *entender* BPE es abrir una palabra y mirar sus pedazos. Tomamos varias —una
común, una técnica, una larguísima, una en otro idioma— y mostramos exactamente en qué subpalabras las
corta el tokenizador. Fíjate en que el espacio inicial suele ir **pegado** a la palabra (« café» no es
lo mismo que «café»): para el modelo, «empezar palabra» es parte del token.


In [ ]:
palabras = ["café", " café", "ingeniería", "tokenización", "anticonstitucionalmente",
            "Kubernetes", "antidisestablishmentarianism", "Aufmerksamkeit"]
for p in palabras:
    ts = trozos(p)
    print(f"{p!r:<28} -> {len(ts):>2} tokens: {ts}")
print("\nUna palabra comun cabe en 1 token; una rara o larga se reparte en varios trozos con sentido.")


## 3. Español frente a inglés: la misma idea cuesta más

Los tokenizadores de los grandes modelos se entrenaron sobre todo con texto en inglés. Por eso el
inglés se «empaqueta» en menos tokens que el español: las palabras y subpalabras inglesas son más
frecuentes en el vocabulario. Comparamos varias frases traducidas y medimos el sobrecoste medio.


In [ ]:
pares = [
    ("Inteligencia artificial para gente curiosa", "Artificial intelligence for curious people"),
    ("El rápido zorro marrón salta sobre el perro perezoso", "The quick brown fox jumps over the lazy dog"),
    ("Necesito una factura con todos los impuestos incluidos", "I need an invoice with all taxes included"),
    ("¿Podrías ayudarme a cambiar mi contraseña, por favor?", "Could you help me change my password, please?"),
]
print(f"{'español':>10} {'tok':>4} | {'inglés':>8} {'tok':>4} | sobrecoste ES")
suma = 0.0
for es, en in pares:
    t_es, t_en = len(tk(es)), len(tk(en))
    extra = 100*(t_es-t_en)/t_en
    suma += extra
    print(f"{es[:8]+'..':>10} {t_es:>4} | {en[:6]+'..':>8} {t_en:>4} | +{extra:.0f}%")
print(f"\nSobrecoste medio del español frente al inglés: +{suma/len(pares):.0f}%  -> mas dinero por la misma tarea.")


## 4. Otros idiomas y otros alfabetos

El efecto se acentúa cuando el idioma usa un alfabeto poco representado en el vocabulario. Comparamos la
misma frase («Buenos días, ¿cómo estás?») en varios idiomas. Cuando el alfabeto no es latino, cada
carácter puede gastar varios tokens, porque se codifica byte a byte.


In [ ]:
frases = {
    "inglés":    "Good morning, how are you?",
    "español":   "Buenos días, ¿cómo estás?",
    "francés":   "Bonjour, comment allez-vous ?",
    "alemán":    "Guten Morgen, wie geht es dir?",
    "ruso":      "Доброе утро, как дела?",
    "griego":    "Καλημέρα, πώς είσαι;",
    "japonés":   "おはよう、元気ですか？",
}
print(f"{'idioma':<10} {'caract.':>8} {'tokens':>7}  tokens/carácter")
for idioma, f in frases.items():
    n = len(tk(f))
    print(f"{idioma:<10} {len(f):>8} {n:>7}  {n/len(f):>6.2f}")
print("\nMismo saludo: el alfabeto latino sale barato; cirilico, griego y japones cuestan mucho mas por caracter.")


## 5. La factura: diez mil tickets de soporte

Caso real: quieres clasificar y resumir los tickets de un mes. Tomamos un ticket típico, contamos sus
tokens de entrada y los de una respuesta breve, y multiplicamos. Usamos precios de **ejemplo** (los
reales varían por modelo y proveedor; cámbialos por los tuyos). La estructura del cálculo es lo que
importa.


In [ ]:
ticket = ("Buenas, llevo desde ayer sin poder acceder a mi cuenta. Me dice que la "
          "contraseña es incorrecta, pero estoy seguro de que es la buena. Ya he probado "
          "a reiniciarla dos veces y no me llega el correo. Necesito entrar hoy porque "
          "tengo que descargar una factura. Gracias.")
respuesta = "Categoría: acceso. Prioridad: alta. Resumen: no recibe el correo de reinicio."
SYSTEM = "Eres un clasificador de tickets. Devuelve categoría, prioridad y resumen."

PRECIO_ENTRADA = 0.50 / 1_000_000     # $ por token de entrada (ejemplo)
PRECIO_SALIDA  = 1.50 / 1_000_000     # $ por token de salida (ejemplo)

t_in = len(tk(SYSTEM)) + len(tk(ticket))
t_out = len(tk(respuesta))
coste_uno = t_in*PRECIO_ENTRADA + t_out*PRECIO_SALIDA
print(f"Un ticket: {t_in} tokens de entrada + {t_out} de salida")
print(f"Coste de un ticket:        ${coste_uno:.6f}")
print(f"Coste de 10.000 tickets:   ${coste_uno*10_000:.2f}")
print(f"Coste de 1.000.000 tickets: ${coste_uno*1_000_000:.2f}  <- a escala, cada token cuenta")


## 6. Entrada frente a salida: la respuesta también se paga (y suele costar más)

En casi todos los proveedores, **un token de salida cuesta más** que uno de entrada. Por eso una
respuesta larga puede dominar la factura aunque la pregunta sea corta. Comparamos la misma tarea
pidiendo una respuesta **breve** (una etiqueta) frente a una **extensa** (un resumen largo y razonado),
sobre los mismos 10.000 tickets.


In [ ]:
resp_breve = "acceso/alta"
resp_larga = ("Categoría: acceso. Prioridad: alta. El usuario no puede iniciar sesión porque "
              "no recibe el correo de reinicio de contraseña. Recomendación: verificar la cola "
              "de correo saliente, confirmar que la dirección no está en lista negra y, mientras "
              "tanto, ofrecer un reinicio manual desde el panel de administración para desbloquear "
              "el acceso hoy mismo, dado que el cliente necesita descargar una factura con urgencia.")
for nombre, r in [("respuesta breve", resp_breve), ("respuesta larga", resp_larga)]:
    to = len(tk(r))
    coste = (t_in*PRECIO_ENTRADA + to*PRECIO_SALIDA) * 10_000
    print(f"{nombre:<16}: {to:>3} tokens de salida -> ${coste:.2f} en 10.000 tickets")
print("\nLa salida se paga, y mas cara: pedir respuestas largas 'por si acaso' multiplica la factura.")


## 7. Comparar modelos: el mismo trabajo, precios muy distintos

No todos los modelos cuestan lo mismo, y la diferencia entre el más caro y el más barato puede ser de
**diez veces o más**. Antes de elegir, conviene poner los números uno al lado del otro. Usamos una
tabla de precios de **ejemplo** (sustitúyela por la de tu proveedor) y calculamos el millón de tickets
con cada uno.


In [ ]:
# Precios de EJEMPLO en $ por millon de tokens (entrada, salida). Cambialos por los reales.
modelos = {
    "económico":  (0.15, 0.60),
    "equilibrado":(0.50, 1.50),
    "premium":    (5.00, 15.00),
}
N = 1_000_000
print(f"{'modelo':<12}{'$/Mtok in':>11}{'$/Mtok out':>12}{'coste 1M tickets':>20}")
costes = {}
for nombre, (pin, pout) in modelos.items():
    coste = t_in*(pin/1e6)*N + t_out*(pout/1e6)*N
    costes[nombre] = coste
    print(f"{nombre:<12}{pin:>11.2f}{pout:>12.2f}{('$'+format(coste,',.0f')):>20}")
print(f"\nEl premium cuesta {costes['premium']/costes['económico']:.0f}x lo que el económico por el MISMO trabajo.")
print("La pregunta no es 'cual es mejor', sino 'cual es suficientemente bueno al menor coste'.")


## 8. El truco de la caché: no pagar dos veces lo mismo

Fíjate en que el `SYSTEM` (las instrucciones) se repite **en cada uno** de los 10.000 tickets. Si el
proveedor cachea ese prefijo común, lo procesa una vez y no en cada llamada. Cuanto más largas sean tus
instrucciones fijas, más pesa cachearlas. Lo medimos con un *system prompt* corto y otro largo, para
ver que el ahorro crece con su longitud.


In [ ]:
SYSTEM_LARGO = SYSTEM + " " + ("Clasifica según estas reglas detalladas: " * 30)
for nombre, sysp in [("system corto", SYSTEM), ("system largo", SYSTEM_LARGO)]:
    ts = len(tk(sysp))
    sin_cache = (ts + len(tk(ticket))) * PRECIO_ENTRADA * 10_000
    con_cache = len(tk(ticket)) * PRECIO_ENTRADA * 10_000 + ts * PRECIO_ENTRADA
    print(f"{nombre} ({ts:>4} tokens): sin caché ${sin_cache:.3f} | con caché ${con_cache:.3f} "
          f"| ahorro {100*(1-con_cache/sin_cache):.0f}%")
print("\nUn system largo repetido 10.000 veces es caro; cachearlo lo convierte en pagarlo una vez.")


## 9. Cuánto ahorra la caché según la longitud del prefijo

El ahorro no es fijo: depende de cuánto pesa el prefijo común frente a lo que cambia en cada llamada
(el ticket). Recorremos prefijos de longitud creciente y dibujamos el porcentaje de ahorro. La curva
deja claro cuándo cachear es anecdótico y cuándo es la diferencia entre viable y ruinoso.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
t_ticket = len(tk(ticket))
longitudes = [10, 50, 100, 200, 400, 800, 1600, 3200]
ahorros = []
for L in longitudes:
    sin_cache = (L + t_ticket) * 10_000
    con_cache = t_ticket * 10_000 + L
    ahorros.append(100*(1 - con_cache/sin_cache))
plt.figure(figsize=(6.2,3.4))
plt.plot(longitudes, ahorros, "-o", color="black")
plt.xlabel("tokens del prefijo fijo (system prompt)"); plt.ylabel("ahorro al cachear (%)")
plt.title("Cuanto mas largo el prefijo comun, mas rinde la cache"); plt.ylim(0,100)
plt.tight_layout(); plt.show()
print("Prefijo corto: ahorro pequeno. Prefijo largo: el ahorro se dispara hacia el limite.")


## 10. La conversación que crece: el historial se paga entero, cada turno

Un detalle que sorprende: un chat no «recuerda» gratis. En cada turno, el modelo vuelve a leer **toda
la conversación anterior** para responder. Así que los tokens de entrada **se acumulan**: el turno 20
paga por los 19 anteriores otra vez. Simulamos una charla y sumamos lo que se procesa de entrada a lo
largo de ella.


In [ ]:
turno_usuario = "Vale, ¿y eso cómo lo aplico a mi caso concreto con un ejemplo paso a paso?"
turno_modelo  = ("Claro. Primero identificas la entrada, luego defines la salida esperada y "
                 "finalmente validas el resultado antes de usarlo en producción, así no propagas errores.")
por_turno = len(tk(turno_usuario)) + len(tk(turno_modelo))
historial = len(tk(SYSTEM))
acumulado = 0
print(f"{'turno':>5} {'historial (tok)':>16} {'entrada acumulada':>18}")
for n in range(1, 21):
    historial += por_turno          # cada turno anade usuario + respuesta al historial
    acumulado += historial          # ...y el modelo reprocesa TODO el historial para responder
    if n in (1, 5, 10, 15, 20):
        print(f"{n:>5} {historial:>16} {acumulado:>18}")
coste_chat = acumulado * PRECIO_ENTRADA
print(f"\nEn 20 turnos se reprocesan {acumulado} tokens de entrada (~${coste_chat:.4f}).")
print("El historial crece y se paga entero cada vez: por eso las conversaciones largas se encarecen solas.")


## 11. La ventana de contexto: cuánto cabe de una vez

Un modelo solo «ve» un número máximo de tokens a la vez (su **ventana de contexto**). Si le mandas un
PDF enorme, puede no caber, y hay que trocearlo (ahí entra el RAG, capítulo 9). Traduzcamos una ventana
a algo tangible: páginas de texto. Usamos la proporción real tokens/palabra del español que medimos
con el ticket.


In [ ]:
tok_por_palabra = len(tk(ticket)) / len(ticket.split())
palabras_por_pagina = 500
print(f"(En español, ~{tok_por_palabra:.2f} tokens por palabra.)\n")
print("ventana (tokens) | páginas aprox. | equivale a...")
referencias = [(8_000, "un artículo"), (32_000, "un capítulo largo"),
               (128_000, "un libro corto"), (1_000_000, "una biblioteca pequeña")]
for ventana, ref in referencias:
    pags = ventana / tok_por_palabra / palabras_por_pagina
    print(f"   {ventana:>9,}   |   {pags:>6.0f}      | {ref}")


## 12. La ventana, dibujada

Las cifras de la tabla, en barras: ayuda a ver el salto enorme entre una ventana de 8k y una de un
millón. Cada barra es cuántas páginas de texto en español caben de una sola vez.


In [ ]:
nombres = [f"{v//1000}k" if v < 1_000_000 else "1M" for v, _ in referencias]
paginas = [v / tok_por_palabra / palabras_por_pagina for v, _ in referencias]
plt.figure(figsize=(6.2,3.4))
plt.bar(nombres, paginas, color="black")
plt.ylabel("páginas que caben (≈500 palabras/pág.)"); plt.xlabel("tamaño de la ventana (tokens)")
plt.title("Cuanto cabe de una vez en la ventana de contexto")
for i, p in enumerate(paginas):
    plt.text(i, p, f"{p:.0f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()


## 13. Pruébalo tú

1. **Mete tu propio texto** (un correo, un capítulo) y cuenta sus tokens *antes* de pagar por
   procesarlo. ¿Cuántas páginas de tu documento caben en una ventana de 128k?
2. **Abre una palabra tuya:** pasa tu apellido, una marca o un tecnicismo por `trozos(...)` y mira en
   cuántos pedazos se parte. ¿Te sorprende?
3. **Compara idiomas:** tokeniza la misma frase en español, inglés y un idioma con otro alfabeto. ¿Cuál
   gasta más? Eso es dinero.
4. **Cambia los precios** por los de tu modelo real y recalcula el millón de tickets. La diferencia
   entre un modelo y otro puede ser de 10×.
5. **Estima tu caché:** si tu *system prompt* tuviera 2.000 tokens y atendieras un millón de llamadas,
   ¿cuánto ahorrarías cacheándolo? (Reutiliza la fórmula de la sección 9.)
6. **Alarga la conversación:** sube a 50 turnos en la sección 10. ¿Cómo crece el coste acumulado, lineal
   o más deprisa? (Pista: el historial crece en cada turno *y* se reprocesa entero.)


## 14. Errores comunes

- **Estimar tokens «a ojo» por palabras.** Te equivocas, sobre todo con código, emojis, URL u otros
  idiomas. Cuéntalos.
- **Olvidar los tokens de salida.** Suelen costar más por token que los de entrada, y una respuesta
  larga puede dominar la factura. No pidas respuestas largas «por si acaso».
- **Creer que el chat recuerda gratis.** En cada turno se reprocesa todo el historial; las
  conversaciones largas se encarecen solas.
- **Ignorar la caché del prefijo.** Repetir un *system prompt* largo en cada llamada es tirar dinero si
  el proveedor permite cachearlo.
- **Elegir modelo sin comparar precios.** El más caro puede costar 10× el más barato por el mismo
  trabajo; muchas veces uno modesto basta.
- **No mirar la ventana de contexto.** Mandar más tokens de los que caben provoca errores o truncado
  silencioso; por eso existen el troceado y el RAG.


## 15. Qué te llevas

- Se paga por **tokens**, no por preguntas, y los tokens **se cuentan**, no se estiman a ojo. Mídelos
  antes de desplegar, y ábrelos cuando algo no cuadre.
- El **idioma importa**: el español gasta más que el inglés, y los alfabetos no latinos, mucho más.
- La **salida** suele costar más que la entrada, y **comparar modelos** puede dividir la factura por
  diez sin perder calidad útil.
- El **historial** de una conversación se paga entero en cada turno; la **ventana de contexto** marca
  cuánto cabe de una vez; y la **caché de prefijo** evita pagar mil veces las mismas instrucciones.
  Estas cifras deciden si una idea es viable.

**Para seguir:** el capítulo 4 te ayuda a elegir modelo sopesando esto; los capítulos 5-6, a decidir
entre nube y local; y el RAG (capítulo 9) es, en parte, una forma de meter en la ventana solo lo justo.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*